# NB2: Zeitreihenmodellierung und Heimspeicher

*Dezentrale Energieressourcen* werden zunehmend eingesetzt, um zentrale Kraftwerke zu ersetzen und die Dekarbonisierung von Energiesystemen zu unterstützen. Mit den jüngsten **Fortschritten** bei der Integration erneuerbarer Energien – insbesondere von PV-Dachanlagen – ist der nächste logische Schritt, diese Systeme mit **Heimspeichern** zu kombinieren und den **lokalen Verbrauch lokal erzeugten Stroms** zu erhöhen.

Aus Sicht der Verbraucherinnen und Verbraucher gibt es zudem Vorteile, die über die bloße Maximierung der Autarkie und die Verringerung des CO2-Fußabdrucks hinausgehen. Steigende Strompreise, sinkende Einspeisevergütungen und fallende Anschaffungskosten von **Batterieenergiespeichersystemen (BESS)** haben günstige Bedingungen für wirtschaftliche Vorteile geschaffen. Dies hat in den letzten Jahren zu einem deutlichen Wachstum des Marktes für Heimspeicher geführt. Aktuelle Kennzahlen finden sich zum Beispiel bei battery-charts](https://battery-charts.de/)..

Da Heimbatterien eine erhebliche betriebliche Flexibilität bieten, ist die Entwicklung geeigneter **Betriebsstrategien** entscheidend, um den größtmöglichen Nutzen aus der Investition zu ziehen.

Dieses Notebook untersucht:

- die Zeitreihensimulation des Betriebs von BESS im Wohnbereich,
- die techno-ökonomische Bewertung von Heimspeichersystemen,
- den Einfluss von Betriebsstrategien auf die Systemleistung (SCI).



In [ ]:
# install suntime package for deepnote workspace
!pip install suntime==1.3.2

In [ ]:
# importing pandas library for data manipulation and analysis
import pandas as pd
import math
# imports für Analysen im 2. Teil der Übung (feed in damping)
from datetime import datetime, timedelta, timezone
from suntime import Sun


In [ ]:
# PLOTLY - plotting options
pd.options.plotting.backend = "plotly"
template = "plotly_white"
# template = "plotly_dark"

## 1. Datensatz und Basisszenario

Für unsere Analyse betrachten wir einen *repräsentativen* Haushalt mit einer PV-Dachanlage. Die Daten stammen aus dem [*Standard Battery Application Profiles (SBAP)*-Paper von Kucevic, Tepe et al.]([https://doi.org/10.1016/j.est.2019.101077) (Open Data), das aufbereitete Profile mehrerer Batteriespeicheranwendungen bereitstellt, die Forschende für Benchmarking und Tests verwenden können.


In [ ]:
# Load and plot the household consumption and PV generation profiles form SBAP - stored in a csv file.
profile = pd.read_csv("../data/household_profile_2025_scaled.csv", index_col=0, parse_dates=True)

fig = profile[["load", "pv"]].plot(
    template=template,
    labels={"value": "Power [kW]", "index": "Time"},
    title="Household load and PV generation (15-min resolution)"
)
fig.show()


Werfen wir zunächst einen Blick auf unser **Basisszenario**, also ein System **ohne Batteriespeicher**.


<div class="alert alert-block alert-success">

**Task 01**

Analysieren Sie die Last- und Erzeugungsprofile (Ausgangsfall: ohne Batterie):

1. Bestimmen Sie aktuelle Werte für den Endkundenstrompreis `electricity_price_eur` und die Einspeisevergütung `feedin_tariff_eur` für eine PV Keinanlage (<10kWp) mit "Teileinspeisung" aus folgenden Quellen: 
- https://www.bundesnetzagentur.de/DE/Fachthemen/ElektrizitaetundGas/ErneuerbareEnergien/EEG_Foerderung/start.html 
- https://www.bdew.de/service/daten-und-grafiken/bdew-strompreisanalyse/





<div class="alert alert-block alert-success">
<b>Hinweis!</b>

- Vergessen Sie nicht, die Achsen des Plots korrekt zu beschriften und eine Legende einzufügen.
- Verwenden Sie klare Variablennamen und achten Sie in allen Berechnungen auf konsistente Einheiten.
</div>


In [ ]:
## Solution Task I.1.
#  Consider 0.372€/kWh for the electricity price and 0.0778 €/kWh for the feed-in tariff.
electricity_price_eur = 0.372
feedin_tariff_eur = 0.0778

<div class="alert alert-block alert-success">

2. Berechnen Sie die Residuallast und speichern Sie sie als neue Spalte im DataFrame `profile`.  
   Visualisieren Sie die Zeitreihe in einem übersichtlichen Plot, der mindestens Folgendes enthält:
   - Last
   - PV-Erzeugung
   - Residuallast
   
   Verwenden Sie aussagekräftige Achsenbeschriftungen, eine Legende und einen Titel.

In [ ]:
## Solution Task I.2.:
# Calculate the residual load, include it as a new column of the dataframe, and plot load/pv/residual.
profile["residual"] = profile["load"] - profile["pv"]
profile["neg_pv"] = profile["pv"] * -1
fig = profile[["load", "neg_pv", "residual"]].plot(
    template=template,
    labels={"value": "Power [kW]", "index": "Time"},
    title="Baseline time series: load, PV generation, and residual load"
)
fig.show()


<div class="alert alert-block alert-info">

3. a) Definieren Sie eine Funktion `electricity_costs(...)` zur Bestimmung der Stromkosten des Haushalts
3. b) Berechnen Sie die Stromkosten des Haushalts, geben Sie das Ergebnis mit zwei Nachkommastellen an!


<div class="alert alert-block alert-success">

beachten Sie die Sampling-Rate der Daten!

In [ ]:

# Calculate electricity costs from grid exchange. Convention: positive = import (buy), negative = export (sell).
def electricity_costs(...
    grid_sell_eur =
    grid_buy_eur =
    total_costs_eur = 
    return total_costs_eur

In [ ]:
costs_base=electricity_costs(profile["residual"], electricity_price_eur, feedin_tariff_eur)
print(f"Electricity costs: {costs_base:.2f} €")

<div class="alert alert-block alert-info">

4. Die Jahresspitzenerzeugung des PV Generators kann als Indikator für die verbaute PV Leistung herangezogen werden - Ermitteln Sie die Spitzenleistung sowie den Tag an dem diese aufritt.



In [ ]:
 
max_pv = 
print(f"Maximum PV generation: {max_pv:.2f} kW")

max_pv_time = 
print(f"Time of maximum PV generation: {max_pv_time}")

<div class="alert alert-block alert-info">

5. Implementieren Sie die Funktion `self_consumption_rate` (SCR).  
   Die formale Definition finden Sie in den Vorlesungsfolien zu den KPIs (SCR/SSR).
   Beachten Sie die Abtastrate der 



In [ ]:
## Solution Task I.4.:
# Self-consumption rate (SCR)
def self_consumption_rate( ):
    feed_in_energy = 
    pv_energy = 
    SCR =
    return SCR


<div class="alert alert-block alert-success">

6. Implementieren Sie die Funktion `self_sufficiency_rate` (SSR).  
   Die formale Definition finden Sie in den Vorlesungsfolien zu den KPIs (SCR/SSR).


In [ ]:
## Solution Task I.5.:
# Self-sufficiency rate (SSR)
def self_sufficiency_rate(grid_profile, load_profile, dt_hours=0.25):
    grid_buy_energy = grid_profile.loc[grid_profile > 0].sum() * dt_hours
    load_energy = load_profile.sum() * dt_hours
    SSR = 1 - (grid_buy_energy / load_energy)
    return SSR


<div class="alert alert-block alert-info">


7. Berechnen Sie:
- den Jahresstromverbrauch des Haushalts (ohne berücksichtigung der PV)
- die Jahres-Stromerzeugung der PV Analge
- die SCR für das Basis-Szenario ohne Speicher 
- die SSR für das Basis-Szenario ohne Speicher



<div class="alert alert-block alert-success">

beachten Sie die Sampling-Rate der Daten!

In [ ]:

 


total_demand =  # annual demand in kWh
total_generation =  # annual generation in kWh
print(f"Total demand:      {total_demand:.0f} kWh") 
print(f"Total generation:  {total_generation:.0f} kWh")

# SCR / SSR (see lecture slides for definitions)
ssr_base = self_sufficiency_rate( )
scr_base = self_consumption_rate( )
print(f"Self-sufficiency (SSR): {ssr_base*100:.2f} %")
print(f"Self-consumption (SCR): {scr_base*100:.2f} %")


<div class="alert alert-block alert-success">

Im folgenden Blick eine Darstellung von Jahresdauerlinie und Leistungs-Histogramm am Netzanschluss

In [ ]:
# Solution (additional visualizations): baseline duration curve and histogram

# Duration curve of residual load (sorted)
residual_sorted = profile["residual"].sort_values(ascending=False)
fig = residual_sorted.reset_index(drop=True).plot(
    template=template,
    labels={"value": "Residual load [kW]", "index": "Sorted timestep"},
    title="Baseline: residual load duration curve"
)
fig.show()

# Histogram of residual load
fig = profile["residual"].plot.hist(
    template=template,
    nbins=60,
    labels={"value": "Residual load [kW]"},
    title="Baseline: residual load histogram"
)
fig.show()


<div class="alert alert-block alert-warning">

## Berantworten Sie nun die Fragen 1 - 4 im Moodle Test

## 2. Greedy-Strategie

Um Solarenergie besser zu nutzen, installiert der Haushalt ein Speichersystem mit folgenden Kennwerten: 
- 5 kWh Nutzkapazität 
- 2 kW maximale Leistung (Lade- / Entladerichtung)
- Produkt- und Installationskosten: 3000 €
- Batterie Lade-/Entladewirkungsgrade von 95 %
- Der "one-way" Wirkungsgrad des Wechselrichters wird zu 99% angenommen
- die Batterie sei leer zu Beginn der Simulation.


Die Batterie benötigt eine Betriebsstrategie: Als ersten Ansatz verwenden wir eine einfache Methode zur Erhöhung des **Eigenverbrauchs**: 

Das Speichersystem "beobachtet" die Residuallast (mittels "Smart Meter"). 
- Immer wenn die PV-Erzeugung die Last übersteigt, wird die Batterie geladen. 
- Immer wenn die PV-Erzeugung unter dem Bedarf liegt, entlädt es, um die Differenz zu decken. 

Diese Strategie versucht, möglichst viel PV-Überschuss zu speichern — daher der Name **Greedy-Strategie**.


In [ ]:
# solution . version mit Dictionary (besser lesbar, übersichtlicher, einfacher zu erweitern)
my_battery = {
    "battery_cost_eur": 3000.0,                 # €
    "capacity_kwh": 5.0,                        # kWh
    "max_power_kw": 2.0,                        # kW
    "battery_one_way_efficiency": 0.95,         # one-way efficiency
    "inverter_one_way_efficiency": 0.99,         # one-way efficiency
    "soe": 0.0                                  # initial state of energy (SoE) in p.u. of capacity (0-1)
}   

<div class="alert alert-block alert-info">

**TASK 2**
Im gleichen Format, aber anstatt des hier gezeigten Beispeicls sollen nun die technischen und ökonomischen Daten der Batterie "Tesla Powerwall 2" (siehe Vorlesungsfolien) als "Dictionary" abgelegt werden.

**Stellen Sie sicher, dass diese Batterie in den folgenden Simulationen verwendet wird!**

In [ ]:
# solution . version mit Dictionary (besser lesbar, übersichtlicher, einfacher zu erweitern)
tesla_powerwall2 = {
    "battery_cost_eur":  }   

<div class="alert alert-block alert-success">

2. Implementieren Sie eine Greedy-Strategie zur Erhöhung des Eigenverbrauchs. Definieren Sie eine Funktion `greedy_strategy()`, die das DataFrame `profile` sowie das dictionary `my_battery` als Eingaben erhält und ein neues DataFrame (**Kopie!**) zurückgibt, welches für jeden Zeitschritt:
   - den Ladezustand der Batterie (SoC),
   - die Speicherleistung und
   - die resultierende Netzleistung enthält.

   Stellen Sie sicher, dass Leistungs- und SoC-Grenzen nicht verletzt werden.


<div class="alert alert-block alert-success">
<b>Hinweis!</b>

- Der Batteriezustand hängt von vorherigen Zeitschritten ab, daher ist eine iterative Simulation erforderlich.
- Für eine bessere Performance sollten Sie wiederholte <code>df.loc[...]</code>-Zuweisungen innerhalb der Schleife vermeiden — sammeln Sie die Ergebnisse stattdessen in Arrays/Listen und weisen Sie sie anschließend gesammelt zu.
- die Effizienz des AC gekoppelten Speichers wird durch die Beiträge von Inverter und Batterie beeinflusst!
- Verwenden Sie im gesamten Notebook konsequent dieselbe Vorzeichenkonvention für positive und negative Leistungen.
</div>


### Vorzeichenkonvention für Speicher-, Residual- und Netzleistung

In diesem Notebook wird für die Betriebsstrategien folgende Konvention verwendet:

- **Batterieleistung `power`**
  - `power > 0`: **Laden** der Batterie
  - `power < 0`: **Entladen** der Batterie
- **Netzleistung `grid`**
  - `grid > 0`: **Netzbezug**
  - `grid < 0`: **Netzeinspeisung**
- **Residualleistung `residual`**
  - `residual > 0`: Lastüberschuss / Importbedarf
  - `residual < 0`: PV-Überschuss / Exporttendenz




In [ ]:
def greedy_strategy(profile, battery):
    """Greedy battery dispatch based on a residual power profile.

    Sign convention:
    - battery power > 0: charging
    - battery power < 0: discharging
    - grid power    > 0: import, < 0: export

    To stay as close as possible to the original NB1 behaviour, the battery
    power is derived from the realized SoE change in each timestep.
    """
    df = profile.copy()

    dt = 0.25  # hours (15 min)

    capacity = float(battery["capacity_kwh"])
    max_power = float(battery["max_power_kw"])
    eta_bat = float(battery["battery_one_way_efficiency"])
    eta_inv = float(battery["inverter_one_way_efficiency"])
    eta_total = eta_bat * eta_inv
    soe = float(battery.get("soe", 0.0))

    residual = df["residual"].to_numpy()
    n = len(df)

    grid_arr = [0.0] * n
    power_arr = [0.0] * n
    soe_arr = [0.0] * n

    for k in range(n):
        r = float(residual[k])

        if r < 0:
            # charge
            p_cmd = min(abs(r), max_power)
            soe_new = min(soe + (p_cmd * dt * eta_total) / capacity, 1.0)
        else:
            # discharge
            p_cmd = min(abs(r), max_power)
            soe_new = max(soe - (p_cmd * dt * (1.0 / eta_total)) / capacity, 0.0)

        # positive -> charge, negative -> discharge
        power = (soe_new - soe) * capacity / dt
        grid = r + power

        soe = soe_new

        grid_arr[k] = grid
        power_arr[k] = power
        soe_arr[k] = soe

    df["grid"] = grid_arr
    df["power"] = power_arr
    df["soe"] = soe_arr

    return df


In [ ]:
df_greedy = greedy_strategy(profile=profile, battery=my_battery) # this can take some time to compute

In [ ]:
# Visualize greedy strategy operation (power)
fig = df_greedy[["residual", "grid", "power"]].plot(
    template=template,
    labels={"value": "Power [kW]", "index": "Time"},
    title="Greedy strategy – power time series"
)
fig.show()


In [ ]:
# Visualize greedy strategy operation (SoC)
fig = df_greedy["soe"].plot(
    template=template,
    labels={"value": "SoE [p.u.]", "index": "Time"},
    title="Greedy strategy – state of energy"
)
fig.show()


<div class="alert alert-block alert-info">

2. Berechnen Sie die resultierenden Kennzahlen SCR und SSR. Vergleichen Sie die Ergebnisse mit dem Basisszenario (ohne Speicher).


In [ ]:
 
# SCR / SSR (see lecture slides for definitions)
ssr_greedy = self_sufficiency_rate( )
scr_greedy = self_consumption_rate( )
print(f"Self-sufficiency (SSR): {ssr_greedy*100:.2f} % ({(ssr_greedy - ssr_base)*100:+.2f}%)")
print(f"Self-consumption (SCR): {scr_greedy*100:.2f} % ({(scr_greedy - scr_base)*100:+.2f}%)")

In [ ]:
# Solution (additional visualizations): greedy strategy duration curves

# Grid power duration curve
grid_sorted = df_greedy["grid"].sort_values(ascending=False)
fig = grid_sorted.reset_index(drop=True).plot(
    template=template,
    labels={"value": "Grid power [kW]", "index": "Sorted timestep"},
    title="Greedy strategy: grid power duration curve"
)
fig.show()

# SoC histogram
fig = df_greedy["soe"].plot.hist(
    template=template,
    nbins=50,
    labels={"value": "SoE [p.u.]"},
    title="Greedy strategy: SoE histogram"
)
fig.show()


<div class="alert alert-block alert-info">

3. Berechnen Sie die resultierenden Stromkosten und die *Einsparungen* im Vergleich zum Referenzszenario.

In [ ]:
# Calculate the electricity costs with the greedy strategy and compare it to the baseline.

costs_greedy = electricity_costs( )
print(f"Electricity costs: {costs_greedy:.2f} € ({costs_greedy-costs_base:.2f} €, {(costs_greedy-costs_base)/costs_base*100:-.2f} %)")


<div class="alert alert-block alert-info">

4. Berechnen Sie, nach wie vielen Jahren sich der Speicher mit dieser simplen Betriebsstrategie amortisiert.

In [ ]:
## simple payback time calculation
investment_costs =  
 
payback_time =  
print(f"Simple payback time: {payback_time:.1f} years")

## 3. Feed-in-Damp-Strategie

Eine zweite Strategie zielt darauf ab, **PV-Einspeisespitzen zu reduzieren**, indem die Batterie bereits vor Sonnenuntergang geladen wird, sodass PV-Überschüsse später am Tag aufgenommen werden können.
Ein zentrales Element ist die Abschätzung der Sonnenuntergangszeit für einen bestimmten Tag **und Standort**.

Hierfür kann das Python-Paket *suntime* verwendet werden.


Die folgende kurze Demo zeigt, wie man *suntime* verwendet, um die Sonnenuntergangszeit für einen gewählten Standort zu bestimmen.


Hier ist eine kurze Demo zur Verwendung von *suntime*.


In [ ]:
# documentation: https://github.com/SatAgro/suntime
# from suntime import Sun
# from datetime import timedelta, timezone

<div class="alert alert-block alert-info">
<b>TASK 5</b>

<ul>
  <li> Testen Sie den Code zur FID Strategie für die vorgegebenen Koordinaten (Kempten) </li>
  <li> Ändern Sie den Standort auf die Standorte: München, Berlin und Hamburg</li>
  <li> Wie sensitiv ist die Feed-in-Damp-Strategie gegenüber den gewählten Standorten (Breiten-/Längengrad)?</li>
  <li> Was passiert an Tagen mit sehr geringer PV-Erzeugung (z. B. Wintertagen)?</li>
  <li> Welche Strategie ist robuster gegenüber Prognosefehlern: <b>greedy</b> oder <b>feed-in damp</b>? Warum?</li>
</ul>
</div>


In [ ]:
# Calculate sunset time for a particular location and date (example values)

location_name = "Kempten"
latitude = 47.7
longitude = 10.31


In [ ]:
# display sunset time for the current day using the same local-time convention as the control law
sun = Sun(latitude, longitude)  # initialize Sun object with location coordinates
local_tz = timezone(timedelta(hours=1.0), name="Europe/Berlin")

today_local_dt = datetime.now(local_tz)
sunset_time_utc = sun.get_sunset_time(today_local_dt)
sunset_time_local = sunset_time_utc.astimezone(local_tz)

print(f"Location:   {location_name} ({latitude:.4f}, {longitude:.4f})")
print(f"UTC sunset time:   {sunset_time_utc}")
print(f"Local sunset time: {sunset_time_local}")


In [ ]:
def add_solar_timing_columns(profile, latitude, longitude, reserve_hours=1.0):
    """Return a copy of `profile` with sunset-related timing columns.

    """
    df = profile.copy()
    sun = Sun(latitude, longitude)
    local_tz = timezone(timedelta(hours=1.0), name="Europe/Berlin")

    sunset_time_local_arr = []
    remaining_sun_hours_arr = []
    remaining_fid_hours_arr = []

    for time in df.index:
        sunset_time_utc = sun.get_sunset_time(time)
        sunset_time_local = sunset_time_utc.astimezone(local_tz).replace(tzinfo=None)

        remaining_sun_hours = (sunset_time_local - time).total_seconds() / 3600.0
        target_time = sunset_time_local - timedelta(hours=reserve_hours)

        # Keep the original timing behaviour from previous lecture definition
        remaining_fid_hours = (target_time - time).seconds / 3600.0

        sunset_time_local_arr.append(sunset_time_local)
        remaining_sun_hours_arr.append(remaining_sun_hours)
        remaining_fid_hours_arr.append(remaining_fid_hours)

    df["sunset_time_local"] = sunset_time_local_arr
    df["remaining_sun_hours"] = remaining_sun_hours_arr
    df["remaining_fid_hours"] = remaining_fid_hours_arr

    return df


# Example: store remaining sun hours for every timestep in a new dataframe
profile_solar = add_solar_timing_columns(profile, latitude, longitude, reserve_hours=1.0)
profile_solar[["remaining_sun_hours", "remaining_fid_hours"]].head()


<div class="alert alert-block alert-success">

Implementieren und analysieren Sie die **Feed-in-Damp-Strategie** für Ihren Standort: Erstellen Sie eine Funktion `feedin_damp(...)` die den DataFrame `profile` sowie das Dictionary `battery` als Eingaben erhält und einen neuen DataFrame mit **Batterie-SoC**, **Batterieleistung** und **Netzleistung** für jeden Zeitschritt zurückgibt. Stellen Sie sicher, dass **Leistungs- und SoE-Grenzen** nicht verletzt werden. Berücksichtigen Sie **Lade-/Entladewirkungsgrade** (vgl. Greedy-Strategie).

- Berechnen Sie **SCR** und **SSR** und bewerten Sie die Veränderung im Vergleich zum **Basisszenario**.

- Berechnen Sie die **Stromkosten des Haushalts** bei Verwendung eines Speichers mit der **FID-Strategie** und bewerten Sie die resultierenden **Einsparungen** im Vergleich zum Basisszenario.

- **TASK (Diskussion):** In welchen Zeiträumen verhält sich die Feed-in-Damp-Strategie anders als die **Greedy-Strategie**? Was ist die beabsichtigte Wirkung auf **PV-Einspeisespitzen**?


In [ ]:
def feedin_damp(profile, battery, latitude, longitude, reserve_hours=1.0):
    """Feed-in-damp battery control.

    Sign convention:
    - battery power > 0: charging
    - battery power < 0: discharging
    - grid power    > 0: import, < 0: export

    The control law is intentionally kept close to the original NB1 implementation.
    Only the battery power sign is translated to:
        positive -> charge
        negative -> discharge
    """
    df = add_solar_timing_columns(profile, latitude, longitude, reserve_hours=reserve_hours)

    dt = 0.25  # hours (15 min)

    capacity = float(battery["capacity_kwh"])
    max_power = float(battery["max_power_kw"])
    eta_bat = float(battery["battery_one_way_efficiency"])
    eta_inv = float(battery["inverter_one_way_efficiency"])
    eta_total = eta_bat * eta_inv
    soe = float(battery.get("soe", 0.0))

    residual = df["residual"].to_numpy()
    remaining_fid_hours = df["remaining_fid_hours"].to_numpy()
    n = len(df)

    grid_arr = [0.0] * n
    power_arr = [0.0] * n
    soe_arr = [0.0] * n

    for k in range(n):
        r = float(residual[k])

        if r < 0:
            energy_remaining = capacity * (1.0 - soe)
            rem_h = float(remaining_fid_hours[k])

            if rem_h > 0:
                power_ref = energy_remaining / rem_h
            else:
                power_ref = energy_remaining / 1.0

            power_ref = min(power_ref, max_power)
            p_cmd = min(abs(r), power_ref)
            soe_new = min(soe + (p_cmd * dt * eta_total) / capacity, 1.0)
        else:
            p_cmd = min(abs(r), max_power)
            soe_new = max(soe - (p_cmd * dt * (1.0 / eta_total)) / capacity, 0.0)

        # positive -> charge, negative -> discharge
        power = (soe_new - soe) * capacity / dt
        grid = r + power

        soe = soe_new

        grid_arr[k] = grid
        power_arr[k] = power
        soe_arr[k] = soe

    df["grid"] = grid_arr
    df["power"] = power_arr
    df["soe"] = soe_arr

    return df


In [ ]:
# Solution Task III.1.: run feed-in damp strategy for the chosen location
df_damp = feedin_damp(
    profile=profile,
    battery=my_battery,
    latitude=latitude,
    longitude=longitude,
    reserve_hours=1.0,
)  # this will also take some time to compute


In [ ]:
# Solution (additional visualizations): feed-in damp strategy duration curves

# Sort duration curves
p_sorted = df_damp["power"].sort_values(ascending=False).reset_index(drop=True)
g_sorted = df_damp["grid"].sort_values(ascending=False).reset_index(drop=True)

# Combine into one dataframe
df_duration = pd.DataFrame({
    "Battery power": p_sorted,
    "Grid power": g_sorted
})

# Plot both curves
fig = df_duration.plot(
    template=template,
    labels={"value": "Power [kW]", "index": "Sorted timestep"},
    title=f"Feed-in damp strategy: duration curves ({location_name})"
)

fig.show()


In [ ]:
# Visualize the resulting operation (include chosen location in the title)
fig = df_damp[["residual", "grid", "power"]].plot(
    template=template,
    labels={"value": "Power [kW]", "index": "Time"},
    title=f"Feed-in damp strategy – power time series ({location_name})"
)
fig.show()


In [ ]:
# Solution Task III.2.:

# SCR / SSR (see lecture slides for definitions)
ssr_damp = self_sufficiency_rate(df_damp["grid"], df_damp["load"], dt_hours=dt_hours)
scr_damp = self_consumption_rate(df_damp["grid"], df_damp["pv"], dt_hours=dt_hours)
print(f"Self-sufficiency (SSR): {ssr_damp*100:.2f} % ({(ssr_damp - ssr_base)/ssr_base*100:+.2f}%)")
print(f"Self-consumption (SCR): {scr_damp*100:.2f} % ({(scr_damp - scr_base)/scr_base*100:+.2f}%)")


In [ ]:
# Solution Task III.3.:

costs_damp = electricity_costs(df_damp["grid"], electricity_price_eur, feedin_tariff_eur, dt_hours=dt_hours)
print(f"Electricity costs: {costs_damp:.2f} € ({costs_damp-costs_base:.2f} €, {(costs_damp-costs_base)/costs_base*100:-.2f} %)")


In [ ]:
## Solution (additional analysis): simple payback time calculation
investment_costs = my_battery['battery_cost_eur']
annual_savings = costs_base - costs_damp
payback_time = investment_costs / annual_savings
print(f"Simple payback time: {payback_time:.1f} years")

Testen Sie jetzt die anderen Standorte

## 4. Vergleich von *Greedy* und *Feed-in Damp*

Wir haben beide Betriebsstrategien zur Erhöhung des Eigenverbrauchs implementiert und analysiert.
Es passiert viel gleichzeitig — vergleichen wir daher beide Ansätze direkt miteinander anhand einiger Visualisierungen.


In [ ]:
# Compare battery power time series (overlay)
dfbat = pd.concat(
    [df_greedy["power"].rename("power - greedy"), df_damp["power"].rename("power - feed-in damp")],
    axis=1
)
fig = dfbat.plot(template=template, labels={"value": "Power [kW]", "index": "Time"}, title="Battery power comparison: greedy vs. feed-in damp")
fig.show()


In [ ]:
dfbat.plot.hist(
    template=template,
    log_y=True,
    labels={"value": "Power [kW]"}
).update_layout(
    barmode="overlay",
    title="Battery power distribution: greedy vs. feed-in damp"
).update_traces(opacity=0.75)

In [ ]:
# Compare SOE time series (overlay)

# dfsoe combines the SoE time series from both strategies (greedy and feed in damping) into one DataFrame for comparison
dfsoe = pd.concat(
    [df_greedy["soe"].rename("SOE - greedy"), df_damp["soe"].rename("SOE - feed-in damp")], 
    axis=1
)
fig = dfsoe.plot(template=template, labels={"value": "SOE [p.u.]", "index": "Time"}, title="SOE comparison: greedy vs. feed-in damp")
fig.show()


In [ ]:
# Compare SoE distributions (histogram)
dfsoe.plot.hist(
    template=template,
    log_y=True,
    labels={"value": "SoE [p.u.]"}
).update_layout(
    barmode="overlay",
    title="SoE Distribution"
).update_traces(opacity=0.75)

In [ ]:
# Compare grid power distributions (histogram)
dfgrid = pd.concat(
    [df_greedy["grid"].rename("grid - greedy"), df_damp["grid"].rename("grid - feed-in damp")],
    axis=1
)
dfgrid.plot.hist(
    template=template,
    log_y=True,
    labels={"value": "Grid power [kW]"}
).update_layout(
    barmode="overlay",
    title="Grid Power Distribution"
).update_traces(opacity=0.75)

<div class="alert alert-block alert-info">
<b> Task 6 </b> 

 Vergleichen Sie beide Strategien. Welche Vorteile ergebne sich aus Sicht des  <b>Haushaltskundin/des Haushaltskunden</b> als auch aus Sicht des <b>Netzbetreibers</b>?. Nutzen Sie die Informationen um dien Moodle TEst abzuschließen!
</div>
